# Calm vs Stress Validation-Regime Experiment

Code for the paper "Silent Failure: Validation-Regime Selection as One Source of Deep-Learning Fragility in Volatility Forecasting".

The experiment varies one thing and holds the rest fixed.

- Train: 2000 to 2012, same for all models.
- Test: 2020 to 2025, frozen, same for all models.
- Calm arm: validate on 2013 to 2014.
- Stress arm: validate on 2015 to 2016.

Six deep-learning models run here. LSTM-base and TFT-base use fixed configs. The LSTM and the TFT are then searched on each arm. Both the LSTM and the TFT select hyperparameters on validation QLIKE. Training uses an MSE loss in both cases. Only the selection signal is QLIKE. The Optuna sampler is seeded, so the calm and stress arms differ only in the validation window.

The four GARCH baselines are fitted in R. Their forecasts are read here from garch_forecasts.csv for the combined ten-model panel. If that file is not present, the notebook skips the combined panel and reports the deep-learning models only.

Run the cells from top to bottom. A GPU is recommended for the TFT.

## 1. Setup, data, and windows

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy import stats
import optuna
import os
import pickle
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
import yfinance as yf

sp500 = yf.download("^GSPC", start="2000-08-01", end="2025-01-22",
                    progress=False, auto_adjust=False)
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)
sp500 = sp500[["Open", "High", "Low", "Close"]].copy()
sp500["returns"] = sp500["Close"].pct_change()
sp500["rv_gk"] = (0.5 * (np.log(sp500["High"] / sp500["Low"])) ** 2
                  - (2 * np.log(2) - 1) * (np.log(sp500["Close"] / sp500["Open"])) ** 2)
sp500 = sp500.dropna()

sp500_dt = sp500.copy()                       # date index, LSTM path
sp500_ti = sp500.reset_index().rename(columns={"Date": "date"})
sp500_ti["time_idx"] = np.arange(len(sp500_ti))
sp500_ti["group"] = "SP500"                   # TFT path

print(f"Total obs: {len(sp500)}  ({sp500.index.min().date()} -> {sp500.index.max().date()})")

# Export the series the GARCH R script reads (date, returns, rv_gk).
# garch_forecasts.R fits the GARCH baselines on this same S&P 500 series.
sp500_dt.reset_index().rename(columns={"Date": "date"})[["date", "returns", "rv_gk"]] \
    .to_csv("sp500_for_garch.csv", index=False)

# Window boundaries (the only thing that defines the experiment)
TRAIN_END   = "2013-01-01"
CALM_START, CALM_END     = "2013-01-01", "2015-01-01"
STRESS_START, STRESS_END = "2015-01-01", "2017-01-01"
TEST_START  = "2020-01-01"

ARMS = {
    "calm":   (CALM_START,   CALM_END),
    "stress": (STRESS_START, STRESS_END),
}

def ann_vol(mask):
    d = sp500_dt.loc[mask, "rv_gk"]
    return np.sqrt(d.mean() * 252) * 100

print(f"Train (<2013) vol:  {ann_vol(sp500_dt.index < TRAIN_END):.2f}%")
print(f"Calm  2013-14 vol:  {ann_vol((sp500_dt.index >= CALM_START)&(sp500_dt.index < CALM_END)):.2f}%")
print(f"Stress 2015-16 vol: {ann_vol((sp500_dt.index >= STRESS_START)&(sp500_dt.index < STRESS_END)):.2f}%")

## 2. Helpers

In [ ]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i - seq_length:i])
        y.append(data[i])
    return np.array(X), np.array(y)

def calculate_qlike(actual, predicted):
    actual = np.array(actual).flatten()
    predicted = np.array(predicted).flatten()
    eps = 1e-10
    predicted = np.maximum(predicted, eps)
    actual = np.maximum(actual, eps)
    return np.mean(actual / predicted - np.log(actual / predicted) - 1)

class LSTMVolatility(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

## 3. LSTM path (selects on QLIKE)

Training and early stopping use MSE. The Optuna objective returns validation QLIKE, so the search selects on QLIKE.

In [ ]:
def lstm_data(val_start, val_end):
    train = sp500_dt[sp500_dt.index < TRAIN_END]
    val   = sp500_dt[(sp500_dt.index >= val_start) & (sp500_dt.index < val_end)]
    test  = sp500_dt[sp500_dt.index >= TEST_START]
    tr = train["rv_gk"].values.reshape(-1, 1)
    va = val["rv_gk"].values.reshape(-1, 1)
    te = test["rv_gk"].values.reshape(-1, 1)
    scaler = StandardScaler()
    tr_s = scaler.fit_transform(tr)              # fit on TRAIN only
    va_s = scaler.transform(va)
    te_s = scaler.transform(te)
    return tr_s, va_s, te_s, scaler

def lstm_train_eval(seq_length, hidden, layers, dropout, lr, batch, val_start, val_end,
                    epochs=100, patience=10, seed=SEED):
    """Train the LSTM with MSE loss and MSE early stopping.
    Returns the test forecast, the best validation MSE, and the validation QLIKE.
    The validation QLIKE is the selection metric used by Optuna."""
    pl.seed_everything(seed, workers=True)
    tr_s, va_s, te_s, scaler = lstm_data(val_start, val_end)
    # history-prepend so val and test sequences are leak-free and full-length
    va_h = np.vstack([tr_s[-seq_length:], va_s])
    te_h = np.vstack([va_s[-seq_length:], te_s])
    Xtr, ytr = create_sequences(tr_s, seq_length)
    Xva, yva = create_sequences(va_h, seq_length)
    Xte, yte = create_sequences(te_h, seq_length)
    Xtr, ytr = torch.FloatTensor(Xtr), torch.FloatTensor(ytr)
    Xva, yva = torch.FloatTensor(Xva), torch.FloatTensor(yva)
    Xte = torch.FloatTensor(Xte)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True)

    model = LSTMVolatility(1, hidden, layers, dropout)
    crit = nn.MSELoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    best, best_state, wait = float("inf"), None, 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vloss = crit(model(Xva), yva).item()
        if vloss < best:
            best, best_state, wait = vloss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    model.eval()

    # validation QLIKE on the original scale: the selection metric
    with torch.no_grad():
        val_pred = model(Xva).numpy()
    val_pred_orig   = scaler.inverse_transform(val_pred).flatten()
    val_actual_orig = scaler.inverse_transform(yva.numpy().reshape(-1, 1)).flatten()
    val_qlike = calculate_qlike(val_actual_orig, val_pred_orig)

    # test forecast on the original variance scale
    with torch.no_grad():
        pred = model(Xte).numpy()
    pred_orig = scaler.inverse_transform(pred).flatten()
    return pred_orig, best, val_qlike

def lstm_objective_factory(val_start, val_end):
    """LSTM Optuna objective. Selects on validation QLIKE."""
    def objective(trial):
        seq_length = trial.suggest_categorical("seq_length", [22, 44, 66])
        hidden     = trial.suggest_categorical("hidden_size", [64, 128])
        layers     = trial.suggest_int("num_layers", 1, 2)
        dropout    = trial.suggest_float("dropout", 0.1, 0.3, step=0.1)
        lr         = trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True)
        batch      = trial.suggest_categorical("batch_size", [32, 64])
        _, _, val_qlike = lstm_train_eval(seq_length, hidden, layers, dropout, lr, batch,
                                          val_start, val_end, epochs=100, patience=10)
        return val_qlike
    return objective

## 4. TFT path (selects on QLIKE)

In [ ]:
MAX_PRED = 1

def tft_build(val_start, val_end, max_encoder):
    train_end_idx = sp500_ti[sp500_ti["date"] < TRAIN_END]["time_idx"].max()
    val_end_idx   = sp500_ti[sp500_ti["date"] < val_end]["time_idx"].max()
    val_start_idx = sp500_ti[sp500_ti["date"] < val_start]["time_idx"].max()
    train_df = sp500_ti[sp500_ti["time_idx"] <= train_end_idx]
    training = TimeSeriesDataSet(
        train_df, time_idx="time_idx", target="rv_gk", group_ids=["group"],
        max_encoder_length=max_encoder, max_prediction_length=MAX_PRED,
        static_categoricals=["group"],
        time_varying_known_reals=["time_idx"],
        time_varying_unknown_reals=["rv_gk"],
        target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    validation = TimeSeriesDataSet.from_dataset(
        training,
        sp500_ti[(sp500_ti["time_idx"] > val_start_idx - max_encoder) &
                 (sp500_ti["time_idx"] <= val_end_idx)],
        stop_randomization=True,
    )
    test_start_idx = sp500_ti[sp500_ti["date"] < TEST_START]["time_idx"].max()
    test_ds = TimeSeriesDataSet.from_dataset(
        training, sp500_ti[sp500_ti["time_idx"] > test_start_idx - max_encoder],
        stop_randomization=True,
    )
    return training, validation, test_ds

def tft_train_eval(cfg, val_start, val_end, epochs=100, patience=15, seed=SEED):
    pl.seed_everything(seed, workers=True)
    training, validation, test_ds = tft_build(val_start, val_end, cfg["max_encoder_length"])
    tdl = training.to_dataloader(train=True, batch_size=cfg["batch_size"], num_workers=0)
    vdl = validation.to_dataloader(train=False, batch_size=cfg["batch_size"], num_workers=0)
    xdl = test_ds.to_dataloader(train=False, batch_size=cfg["batch_size"], num_workers=0)
    model = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=cfg["learning_rate"], hidden_size=cfg["hidden_size"],
        attention_head_size=cfg["attention_head_size"], dropout=cfg["dropout"],
        hidden_continuous_size=cfg["hidden_continuous_size"],
        lstm_layers=cfg.get("lstm_layers", 1),
        output_size=1, loss=RMSE(), log_interval=-1, reduce_on_plateau_patience=3,
    )
    es = EarlyStopping(monitor="val_loss", patience=patience, mode="min")
    trainer = pl.Trainer(max_epochs=epochs, accelerator="auto", devices=1,
                         gradient_clip_val=0.1, callbacks=[es],
                         enable_progress_bar=False, enable_model_summary=False, logger=False)
    trainer.fit(model, tdl, vdl)
    # validation QLIKE: the selection metric
    vpred = model.predict(vdl).cpu().numpy().astype(np.float64)
    vact  = torch.cat([y[0] for x, y in iter(vdl)]).cpu().numpy().astype(np.float64)
    val_qlike = calculate_qlike(vact, vpred)
    # test forecast on the original variance scale
    tpred = model.predict(xdl).cpu().numpy().astype(np.float64).flatten()
    return tpred, val_qlike

def tft_objective_factory(val_start, val_end):
    def objective(trial):
        cfg = {
            "hidden_size": trial.suggest_categorical("hidden_size", [16, 32, 48, 64]),
            "attention_head_size": trial.suggest_categorical("attention_head_size", [1, 2, 4]),
            "dropout": trial.suggest_float("dropout", 0.05, 0.4, step=0.05),
            "hidden_continuous_size": trial.suggest_categorical("hidden_continuous_size", [8, 16, 32]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
            "max_encoder_length": trial.suggest_categorical("max_encoder_length", [10, 22, 44]),
            "lstm_layers": trial.suggest_int("lstm_layers", 1, 2),
            "batch_size": 32,
        }
        try:
            _, val_qlike = tft_train_eval(cfg, val_start, val_end, epochs=100, patience=15)
        except Exception as e:
            print(f"Trial {trial.number} failed: {e}")
            raise optuna.TrialPruned()
        return val_qlike
    return objective

## 5. Run the search matrix

Two baselines with fixed configs, then four searches: LSTM and TFT on each arm. All four searches select on validation QLIKE. The forecasts and selected hyperparameters are banked to calm_vs_stress_full.pkl.

In [ ]:
def make_study():
    return optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    )

N_TRIALS = 30
test_forecasts = {}   # model name -> test-set variance forecast (original scale)

# Baselines (fixed configs, train 2000-2012), no search
print("=== LSTM baseline ===")
test_forecasts["lstm_base"], _, _ = lstm_train_eval(
    seq_length=22, hidden=32, layers=1, dropout=0.0, lr=0.001, batch=32,
    val_start=CALM_START, val_end=CALM_END, epochs=100, patience=15)

print("=== TFT baseline ===")
tft_base_cfg = dict(hidden_size=32, attention_head_size=4, dropout=0.1,
                    hidden_continuous_size=16, lstm_layers=1,
                    learning_rate=0.001, max_encoder_length=22, batch_size=32)
test_forecasts["tft_base"], _ = tft_train_eval(tft_base_cfg, CALM_START, CALM_END,
                                               epochs=100, patience=15)

# Searches: both LSTM and TFT select on validation QLIKE
best_params = {}
for arm, (vs, ve) in ARMS.items():
    print(f"=== LSTM-opt [{arm}] (QLIKE-selected) ===")
    s = make_study(); s.optimize(lstm_objective_factory(vs, ve), n_trials=N_TRIALS)
    best_params[f"lstm_opt_{arm}"] = s.best_params
    bp = s.best_params
    test_forecasts[f"lstm_opt_{arm}"], _, _ = lstm_train_eval(
        bp["seq_length"], bp["hidden_size"], bp["num_layers"], bp["dropout"],
        bp["learning_rate"], bp["batch_size"], vs, ve, epochs=100, patience=10)

    print(f"=== TFT-opt [{arm}] (QLIKE-selected) ===")
    s = make_study(); s.optimize(tft_objective_factory(vs, ve), n_trials=N_TRIALS)
    best_params[f"tft_opt_{arm}"] = s.best_params
    cfg = dict(s.best_params); cfg["batch_size"] = 32
    test_forecasts[f"tft_opt_{arm}"], _ = tft_train_eval(cfg, vs, ve, epochs=100, patience=15)

print("\nBest params per arm:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# Bank the expensive results so a crash cannot cost the searches again
with open("calm_vs_stress_full.pkl", "wb") as f:
    pickle.dump({"test_forecasts": test_forecasts, "best_params": best_params}, f)
print("\nForecasts banked to calm_vs_stress_full.pkl")

## 6. Evaluation panel (33/67 regime split)

QLIKE accuracy, correlation, and VaR(99%) with Kupiec and Christoffersen tests, split into low, medium, and high volatility terciles on the test actual variance.

In [ ]:
test_dt = sp500_dt[sp500_dt.index >= TEST_START]
actual_full     = test_dt["rv_gk"].values
returns_full    = test_dt["returns"].values
test_dates_full = test_dt.index

# History-prepend keeps the forecasts full-length. Align to a common length to be safe.
L_common = min(len(f) for f in test_forecasts.values())
actual       = actual_full[-L_common:]
returns_test = returns_full[-L_common:]
test_dates   = test_dates_full[-L_common:]
for k in test_forecasts:
    test_forecasts[k] = test_forecasts[k][-L_common:]

# Regime terciles (33/67) on the test actual variance
rv_33 = np.quantile(actual, 0.33)
rv_67 = np.quantile(actual, 0.67)
regime = np.asarray(pd.cut(actual, bins=[0, rv_33, rv_67, np.inf],
                           labels=["Low", "Medium", "High"]))
high_mask = (regime == "High")
low_mask  = (regime == "Low")
print(f"Regime thresholds: 33rd={rv_33:.2e}  67th={rv_67:.2e}")
print(f"High days={high_mask.sum()}  Low days={low_mask.sum()}  N={L_common}")

z_99 = stats.norm.ppf(0.01)   # -2.326

def kupiec_test(n_violations, n_obs, alpha=0.01):
    er = alpha; ar = n_violations / n_obs
    if n_violations == 0:        lr = -2 * n_obs * np.log(1 - er)
    elif n_violations == n_obs:  lr = -2 * n_obs * np.log(er)
    else: lr = -2 * (n_violations * np.log(er / ar) +
                     (n_obs - n_violations) * np.log((1 - er) / (1 - ar)))
    p = 1 - stats.chi2.cdf(lr, df=1)
    return ar, p, p >= 0.05

def christoffersen_test(violations, alpha=0.01):
    v = np.asarray(violations).astype(int)
    n_obs = len(v); n_v = v.sum(); er = alpha; ar = n_v / n_obs
    if n_v == 0:        lr_uc = -2 * n_obs * np.log(1 - er)
    elif n_v == n_obs:  lr_uc = -2 * n_obs * np.log(er)
    else: lr_uc = -2 * (n_v * np.log(er / ar) + (n_obs - n_v) * np.log((1 - er) / (1 - ar)))
    n00 = ((1 - v[:-1]) & (1 - v[1:])).sum(); n01 = ((1 - v[:-1]) & v[1:]).sum()
    n10 = (v[:-1] & (1 - v[1:])).sum();       n11 = (v[:-1] & v[1:]).sum()
    n_0 = n00 + n01; n_1 = n10 + n11
    if n_0 == 0 or n_1 == 0 or n01 == 0 or n10 == 0:
        lr_ind = 0
    else:
        pi01 = n01 / n_0; pi11 = n11 / n_1; pi = (n01 + n11) / (n_0 + n_1)
        lr_ind = -2 * (n00*np.log(1-pi) + n01*np.log(pi) + n10*np.log(1-pi) + n11*np.log(pi)
                       - n00*np.log(1-pi01) - n01*np.log(pi01)
                       - n10*np.log(1-pi11) - n11*np.log(pi11))
    p = 1 - stats.chi2.cdf(lr_uc + lr_ind, df=2)
    return p, p >= 0.05

rows = []
for name, pred in test_forecasts.items():
    pred = np.maximum(pred, 1e-10)
    qlike_all  = calculate_qlike(actual, pred)
    qlike_high = calculate_qlike(actual[high_mask], pred[high_mask])
    qlike_low  = calculate_qlike(actual[low_mask], pred[low_mask])
    c_all  = np.corrcoef(actual, pred)[0, 1]
    c_high = np.corrcoef(actual[high_mask], pred[high_mask])[0, 1]
    c_low  = np.corrcoef(actual[low_mask],  pred[low_mask])[0, 1]
    vol = np.sqrt(pred); var = z_99 * vol
    viol = returns_test < var
    n_pool = int(viol.sum()); n_high = int(viol[high_mask].sum()); n_low = int(viol[low_mask].sum())
    _, kup_p, kup_pass = kupiec_test(n_pool, L_common)
    chr_p, chr_pass = christoffersen_test(viol)
    rows.append({
        "model": name,
        "QLIKE_all": qlike_all, "QLIKE_low": qlike_low, "QLIKE_high": qlike_high,
        "QLIKE_ratio": qlike_high / qlike_low,
        "corr_all": c_all, "corr_high": c_high, "corr_low": c_low,
        "VaR_viol_pool": n_pool, "VaR_rate_pool": n_pool / L_common,
        "VaR_viol_high": n_high, "VaR_viol_low": n_low,
        "Kupiec_p": kup_p, "Kupiec_pass": kup_pass,
        "Christoffersen_p": chr_p, "Christoffersen_pass": chr_pass,
    })
res = pd.DataFrame(rows).set_index("model")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("\n=== QLIKE (accuracy) ===")
print(res[["QLIKE_all","QLIKE_low","QLIKE_high","QLIKE_ratio"]].to_string())
print("\n=== Correlation (tracking) ===")
print(res[["corr_all","corr_high","corr_low"]].to_string())
print("\n=== VaR(99%) violations and tests ===")
print(res[["VaR_viol_pool","VaR_viol_high","VaR_viol_low","Kupiec_p","Christoffersen_p"]].to_string())
res.to_csv("calm_vs_stress_results.csv")
print("\nSaved calm_vs_stress_results.csv")

## 7. Calm vs stress: the headline read

In [ ]:
print("=== FRAGILITY: calm-val vs stress-val (QLIKE-selected models) ===")
for fam in ["lstm_opt", "tft_opt"]:
    c = res.loc[f"{fam}_calm"]; s = res.loc[f"{fam}_stress"]
    print(f"\n{fam}:")
    print(f"  QLIKE_high   calm={c.QLIKE_high:.4f}  stress={s.QLIKE_high:.4f}  "
          f"gap(calm-stress)={c.QLIKE_high - s.QLIKE_high:+.4f}")
    print(f"  QLIKE_ratio  calm={c.QLIKE_ratio:.3f}  stress={s.QLIKE_ratio:.3f}")
    print(f"  VaR viol high-vol  calm={int(c.VaR_viol_high)}  stress={int(s.VaR_viol_high)}")
    print(f"  corr high-vol      calm={c.corr_high:.4f}  stress={s.corr_high:.4f}")

print("\nBaseline reference (no search, matched 2000-2012 train):")
print(res.loc[["lstm_base", "tft_base"], ["QLIKE_ratio", "QLIKE_high", "VaR_viol_high"]].to_string())

## 8. Combined ten-model panel (deep learning + GARCH)

Reads the four GARCH baseline forecasts from garch_forecasts.csv and merges them with the six deep-learning forecasts. If the CSV is not present, this section is skipped and only the deep-learning models are carried forward.

In [ ]:
GARCH_CSV = "garch_forecasts.csv"   # GARCH baseline forecasts, fitted in R

if not os.path.exists(GARCH_CSV):
    print(f"'{GARCH_CSV}' not found. Skipping the combined ten-model panel.")
    print("Add the GARCH forecast CSV to the repo root to run this section.")
    all_forecasts = dict(test_forecasts)   # deep learning only
else:
    g = pd.read_csv(GARCH_CSV)
    print("GARCH CSV:", g.shape, "| columns:", list(g.columns))

    garch_forecasts = {
        "garch_n_static": g["garch_norm_static"].to_numpy(),
        "gjr_t_static":   g["gjr_t_static"].to_numpy(),
        "garch_n_roll":   g["garch_norm_roll"].to_numpy(),
        "gjr_t_roll":     g["gjr_t_roll"].to_numpy(),
    }
    garch_actual  = g["actual_rv_gk"].to_numpy()
    garch_returns = g["returns_test"].to_numpy()

    # Alignment check: GARCH and deep-learning forecasts must be on the same test days
    assert len(garch_actual) == L_common, f"GARCH n={len(garch_actual)} vs DL L_common={L_common}"
    mismatch = np.max(np.abs(garch_actual - actual))
    print(f"Actual-vol alignment: max abs diff = {mismatch:.2e} "
          f"({'OK' if mismatch < 1e-10 else 'WARNING, days do not match'})")
    rmis = np.max(np.abs(garch_returns - returns_test))
    print(f"Returns alignment:    max abs diff = {rmis:.2e} "
          f"({'OK' if rmis < 1e-10 else 'WARNING'})")

    all_forecasts = {}
    all_forecasts.update(test_forecasts)   # 6 deep learning
    all_forecasts.update(garch_forecasts)  # 4 GARCH
    print("Total models:", len(all_forecasts), "->", list(all_forecasts.keys()))

    def qlike(a, p):
        a = np.maximum(np.asarray(a).flatten(), 1e-10); p = np.maximum(np.asarray(p).flatten(), 1e-10)
        return np.mean(a / p - np.log(a / p) - 1)

    def kupiec(viol, alpha=0.01):
        n_obs = len(viol); n_v = int(viol.sum()); er = alpha; ar = n_v / n_obs
        if n_v == 0:        lr = -2 * n_obs * np.log(1 - er)
        elif n_v == n_obs:  lr = -2 * n_obs * np.log(er)
        else: lr = -2 * (n_v*np.log(er/ar) + (n_obs-n_v)*np.log((1-er)/(1-ar)))
        p = 1 - stats.chi2.cdf(lr, df=1)
        return ar, p, p >= 0.05

    def christoffersen(viol, alpha=0.01):
        v = np.asarray(viol).astype(int); n_obs = len(v); n_v = v.sum(); er = alpha; ar = n_v/n_obs
        if n_v == 0:        lr_uc = -2*n_obs*np.log(1-er)
        elif n_v == n_obs:  lr_uc = -2*n_obs*np.log(er)
        else: lr_uc = -2*(n_v*np.log(er/ar) + (n_obs-n_v)*np.log((1-er)/(1-ar)))
        n00=((1-v[:-1])&(1-v[1:])).sum(); n01=((1-v[:-1])&v[1:]).sum()
        n10=(v[:-1]&(1-v[1:])).sum();     n11=(v[:-1]&v[1:]).sum()
        n_0=n00+n01; n_1=n10+n11
        if n_0==0 or n_1==0 or n01==0 or n10==0: lr_ind = 0
        else:
            pi01=n01/n_0; pi11=n11/n_1; pi=(n01+n11)/(n_0+n_1)
            lr_ind=-2*(n00*np.log(1-pi)+n01*np.log(pi)+n10*np.log(1-pi)+n11*np.log(pi)
                       -n00*np.log(1-pi01)-n01*np.log(pi01)-n10*np.log(1-pi11)-n11*np.log(pi11))
        p = 1 - stats.chi2.cdf(lr_uc + lr_ind, df=2)
        return p, p >= 0.05

    rows = []
    for name, pred in all_forecasts.items():
        pred = np.maximum(np.asarray(pred, dtype=np.float64), 1e-10)
        q_all  = qlike(actual, pred)
        q_high = qlike(actual[high_mask], pred[high_mask])
        q_low  = qlike(actual[low_mask],  pred[low_mask])
        c_all  = np.corrcoef(actual, pred)[0, 1]
        c_high = np.corrcoef(actual[high_mask], pred[high_mask])[0, 1]
        c_low  = np.corrcoef(actual[low_mask],  pred[low_mask])[0, 1]
        vol = np.sqrt(pred); var = z_99 * vol; viol = returns_test < var
        n_pool = int(viol.sum()); n_high = int(viol[high_mask].sum()); n_low = int(viol[low_mask].sum())
        _, kp_pool, _ = kupiec(viol)
        cp_pool, _ = christoffersen(viol)
        _, kp_high, _ = kupiec(viol[high_mask])
        _, kp_low,  _ = kupiec(viol[low_mask])
        cp_high, _ = christoffersen(viol[high_mask])
        cp_low,  _ = christoffersen(viol[low_mask])
        rows.append({
            "model": name,
            "QLIKE_all": q_all, "QLIKE_low": q_low, "QLIKE_high": q_high,
            "QLIKE_ratio": q_high / q_low,
            "corr_all": c_all, "corr_high": c_high, "corr_low": c_low,
            "VaR_pool": n_pool, "VaR_high": n_high, "VaR_low": n_low,
            "Kupiec_p_pool": kp_pool, "Kupiec_p_high": kp_high, "Kupiec_p_low": kp_low,
            "Christ_p_pool": cp_pool, "Christ_p_high": cp_high, "Christ_p_low": cp_low,
        })
    res_combined = pd.DataFrame(rows).set_index("model")
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    order = ["lstm_base","tft_base","lstm_opt_calm","tft_opt_calm","lstm_opt_stress","tft_opt_stress",
             "garch_n_static","gjr_t_static","garch_n_roll","gjr_t_roll"]
    res_combined = res_combined.reindex([m for m in order if m in res_combined.index])

    print("\n=== QLIKE (accuracy) ===")
    print(res_combined[["QLIKE_all","QLIKE_low","QLIKE_high","QLIKE_ratio"]].to_string())
    print("\n=== Correlation (tracking) ===")
    print(res_combined[["corr_all","corr_high","corr_low"]].to_string())
    print(f"\n=== VaR violations (of {L_common} / {int(high_mask.sum())} / {int(low_mask.sum())}) ===")
    print(res_combined[["VaR_pool","VaR_high","VaR_low"]].to_string())
    print("\n=== Kupiec p (pass if >= 0.05) ===")
    print(res_combined[["Kupiec_p_pool","Kupiec_p_high","Kupiec_p_low"]].to_string())
    print("\n=== Christoffersen p (pass if >= 0.05; regime split may be underpowered) ===")
    print(res_combined[["Christ_p_pool","Christ_p_high","Christ_p_low"]].to_string())

    res_combined.to_csv("combined_10model_results.csv")
    print("\nSaved combined_10model_results.csv")

## 9. Bootstrap confidence intervals

Regime-stratified resampling of the test days. The key output is whether the calm and stress high-volatility QLIKE intervals overlap.

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(42)
high_idx = np.where(high_mask)[0]
low_idx  = np.where(low_mask)[0]

def boot_metric(fn):
    out = {m: [] for m in all_forecasts}
    for _ in range(N_BOOT):
        hi = rng.choice(high_idx, size=len(high_idx), replace=True)
        lo = rng.choice(low_idx,  size=len(low_idx),  replace=True)
        for m, pred in all_forecasts.items():
            out[m].append(fn(m, pred, hi, lo))
    return {m: np.array(v) for m, v in out.items()}

def _qlike(a, p):
    a = np.maximum(np.asarray(a).flatten(), 1e-10); p = np.maximum(np.asarray(p).flatten(), 1e-10)
    return np.mean(a / p - np.log(a / p) - 1)

def f_qlike_ratio(m, pred, hi, lo): return _qlike(actual[hi], pred[hi]) / _qlike(actual[lo], pred[lo])
def f_qlike_high(m, pred, hi, lo):  return _qlike(actual[hi], pred[hi])
def f_corr_high(m, pred, hi, lo):   return np.corrcoef(actual[hi], pred[hi])[0, 1]

boot_ratio = boot_metric(f_qlike_ratio)
boot_qhigh = boot_metric(f_qlike_high)
boot_chigh = boot_metric(f_corr_high)

def ci(arr): return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

order = [m for m in ["lstm_base","tft_base","lstm_opt_calm","tft_opt_calm",
                     "lstm_opt_stress","tft_opt_stress",
                     "garch_n_static","gjr_t_static","garch_n_roll","gjr_t_roll"]
         if m in all_forecasts]

print("=== QLIKE_high, point + 95% CI ===")
for m in order:
    pt = np.mean(boot_qhigh[m]); lo, hi = ci(boot_qhigh[m])
    print(f"  {m:16s} {pt:6.3f}  [{lo:6.3f}, {hi:6.3f}]")

print("\n=== KEY: calm vs stress, QLIKE_high CIs (non-overlap = significant) ===")
for fam in ["lstm_opt", "tft_opt"]:
    if f"{fam}_calm" in all_forecasts and f"{fam}_stress" in all_forecasts:
        c_lo, c_hi = ci(boot_qhigh[f"{fam}_calm"])
        s_lo, s_hi = ci(boot_qhigh[f"{fam}_stress"])
        overlap = not (c_lo > s_hi or s_lo > c_hi)
        print(f"  {fam}: calm [{c_lo:.3f},{c_hi:.3f}] vs stress [{s_lo:.3f},{s_hi:.3f}] "
              f"-> {'OVERLAP (not sig)' if overlap else 'NON-OVERLAP (significant)'}")

## 10. Diebold-Mariano tests

Regime-conditioned DM with the Harvey-Leybourne-Newbold small-sample correction, for both model families. Includes a high-volatility test, a full-sample test, and a 2020-21 versus 2022-25 re-slice to check the result is not a COVID artefact.

In [ ]:
def qlike_vec(a, p, eps=1e-10):
    a = np.maximum(np.asarray(a, float).flatten(), eps)
    p = np.maximum(np.asarray(p, float).flatten(), eps)
    return a/p - np.log(a/p) - 1.0

def dm_test(l1, l2, h=1):
    """Diebold-Mariano with Harvey-Leybourne-Newbold small-sample correction.
    l1 and l2 are per-day loss series. Positive mean_diff means l1 is worse than l2."""
    d = (np.asarray(l1, float) - np.asarray(l2, float))
    d = d[np.isfinite(d)]; nn = d.size; db = d.mean()
    L = max(h-1, int(np.floor(4*(nn/100.0)**(2/9))))
    lrv = np.sum((d-db)**2)/nn
    for k in range(1, L+1):
        ck = np.sum((d[k:]-db)*(d[:-k]-db))/nn; lrv += 2*(1-k/(L+1))*ck
    dm = db/np.sqrt(lrv/nn)
    hln = np.sqrt((nn+1-2*h+h*(h-1)/nn)/nn); dmc = dm*hln
    return {"dm": dmc, "p": 2*(1-stats.t.cdf(abs(dmc), df=nn-1)), "mean_diff": db, "n": nn}

yr   = np.asarray(test_dates.year)
hv67 = actual >= np.quantile(actual, 0.67)
full = np.ones(L_common, dtype=bool)

for fam in ["tft_opt", "lstm_opt"]:
    calm_fc   = test_forecasts[f"{fam}_calm"]
    stress_fc = test_forecasts[f"{fam}_stress"]
    print(f"\n===== {fam}: Diebold-Mariano (calm vs stress) =====")
    for label, m in [("HIGH-VOL", hv67), ("full sample", full)]:
        a = actual[m]
        lc = qlike_vec(a, calm_fc[m]); ls = qlike_vec(a, stress_fc[m])
        r = dm_test(lc, ls)
        print(f"  {label} (n={r['n']}): QLIKE calm={lc.mean():.4f} stress={ls.mean():.4f} "
              f"diff(calm-stress)={r['mean_diff']:+.4f} | DM={r['dm']:.3f} p={r['p']:.4g} "
              f"-> {'SIGNIFICANT' if r['p']<0.05 else 'n.s.'}")
    for nm, pm in [("2020-21", yr <= 2021), ("2022-25", yr >= 2022)]:
        m = pm & hv67; a = actual[m]
        print(f"  {nm}: high-vol n={int(m.sum()):3d} | QLIKE calm={qlike_vec(a, calm_fc[m]).mean():.3f} "
              f"stress={qlike_vec(a, stress_fc[m]).mean():.3f}")

## 11. Threshold robustness (33/67 and 25/75)

Repeats the high-volatility QLIKE gap and the DM test at a wider 25/75 cut to show the effect does not depend on the tercile choice.

In [ ]:
for fam in ["lstm_opt", "tft_opt"]:
    calm_fc   = test_forecasts[f"{fam}_calm"]
    stress_fc = test_forecasts[f"{fam}_stress"]
    print(f"\n===== {fam}: threshold robustness =====")
    for lo_q, hi_q in [(33, 67), (25, 75)]:
        qlo, qhi = np.percentile(actual, lo_q), np.percentile(actual, hi_q)
        low_m, high_m = actual <= qlo, actual >= qhi
        lc = qlike_vec(actual, calm_fc)[high_m]
        ls = qlike_vec(actual, stress_fc)[high_m]
        r = dm_test(lc, ls)
        print(f"  {lo_q}/{hi_q} (n_high={int(high_m.sum())}): high-vol QLIKE "
              f"calm={lc.mean():.3f} stress={ls.mean():.3f} gap={lc.mean()-ls.mean():+.3f} | "
              f"DM={r['dm']:.2f} p={r['p']:.4g}")

## 12. Figures

In [ ]:
# Figure: the two validation windows
win = sp500_dt.loc["2012-06-01":"2016-12-31"].copy()
win["ann_vol"] = np.sqrt(win["rv_gk"]) * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(win.index, win["ann_vol"], color="#444444", lw=0.7)
ax.axvspan(pd.Timestamp("2013-01-01"), pd.Timestamp("2014-12-31"), color="#4C72B0", alpha=0.18,
           label="Calm validation (2013-2014)")
ax.axvspan(pd.Timestamp("2015-01-01"), pd.Timestamp("2016-12-31"), color="#C44E52", alpha=0.18,
           label="Stressed validation (2015-2016)")
ax.set_ylabel("Annualized volatility (%)")
ax.set_title("The two validation windows: similar typical level, different stress exposure")
ax.legend(loc="upper left", fontsize=8, frameon=False)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig_validation_windows.png", dpi=200, bbox_inches="tight")
print("saved fig_validation_windows.png")
plt.show()

In [ ]:
# Figure: magnitude not tracking (calm-validated TFT under-scales in high vol)
calm   = np.asarray(test_forecasts["tft_opt_calm"],   float)
stress = np.asarray(test_forecasts["tft_opt_stress"], float)
hi = actual >= np.quantile(actual, 0.67)

def blk(name, fc):
    a, f_ = actual[hi], fc[hi]; corr = np.corrcoef(f_, a)[0, 1]
    print(f"{name:7s} high-vol: corr={corr:.3f} | mean fc={f_.mean():.3e} vs realised={a.mean():.3e} "
          f"(ratio {f_.mean()/a.mean():.2f}) | SD fc={f_.std():.3e} vs realised={a.std():.3e} "
          f"(ratio {f_.std()/a.std():.2f})")
    return corr

print(f"high-vol days={int(hi.sum())}\n")
c_calm   = blk("calm", calm)
c_stress = blk("stress", stress)

a = actual[hi]; lim = max(a.max(), calm[hi].max(), stress[hi].max()) * 1.05
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True, sharey=True)
for axi, fc, ttl, cc in [(ax[0], calm[hi], "Calm-validated TFT", c_calm),
                          (ax[1], stress[hi], "Stress-validated TFT", c_stress)]:
    axi.scatter(fc, a, s=14, alpha=0.45, edgecolor="none")
    axi.plot([0, lim], [0, lim], "k--", lw=1, label="perfect scale")
    axi.set_xlim(0, lim); axi.set_ylim(0, lim)
    axi.set_xlabel("Forecast variance"); axi.set_title(f"{ttl}\n(corr = {cc:.2f})")
    axi.legend(loc="upper left", fontsize=8, frameon=False)
ax[0].set_ylabel("Realised variance")
fig.suptitle("High-volatility regime: both track realised variance, but the calm model under-scales",
             fontsize=11)
fig.tight_layout()
fig.savefig("fig_magnitude_not_tracking.png", dpi=200, bbox_inches="tight")
print("\nsaved fig_magnitude_not_tracking.png")
plt.show()

In [ ]:
# Figure: the calm-validation penalty scales with model capacity
# Gap = high-vol QLIKE (calm minus stress) at 33/67, read from the results table.
gap_lstm = res.loc["lstm_opt_calm", "QLIKE_high"] - res.loc["lstm_opt_stress", "QLIKE_high"]
gap_tft  = res.loc["tft_opt_calm",  "QLIKE_high"] - res.loc["tft_opt_stress",  "QLIKE_high"]

models = ["LSTM\n(smaller)", "TFT\n(larger)"]
gaps = [gap_lstm, gap_tft]

fig, ax = plt.subplots(figsize=(5, 4.2))
bars = ax.bar(models, gaps, color=["#4C72B0", "#C44E52"], width=0.55)
for b, gp in zip(bars, gaps):
    ax.text(b.get_x()+b.get_width()/2, gp + max(gaps)*0.03, f"+{gp:.2f}", ha="center", fontsize=11)
ax.set_ylabel("High-volatility QLIKE penalty\n(calm minus stress)")
ax.set_title("The calm-validation penalty scales with model capacity")
ax.set_ylim(0, max(gaps)*1.18)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout()
fig.savefig("fig_capacity_scaling.png", dpi=200, bbox_inches="tight")
print(f"LSTM gap={gap_lstm:+.3f}  TFT gap={gap_tft:+.3f}")
print("saved fig_capacity_scaling.png")
plt.show()

## 13. Validation-window statistics

In [ ]:
def window_stats(label, start, end):
    w = sp500_dt[(sp500_dt.index >= start) & (sp500_dt.index < end)]
    av = np.sqrt(w["rv_gk"]) * np.sqrt(252) * 100
    px = w["Close"]; dd = (px / px.cummax() - 1).min()
    return {"window": label, "days": len(w),
            "mean_ann_vol_pct": round(av.mean(), 1),
            "median_ann_vol_pct": round(av.median(), 1),
            "max_ann_vol_pct": round(av.max(), 1),
            "index_max_drawdown_pct": round(dd*100, 1)}

rows = [window_stats("Calm (2013-2014)", CALM_START, CALM_END),
        window_stats("Stressed (2015-2016)", STRESS_START, STRESS_END)]
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
tab.to_csv("validation_window_stats.csv", index=False)
print("\nSaved validation_window_stats.csv")